In [ ]:
!pip install scikit-optimize

In [ ]:
%%time
# Imports principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from skimage.io import imread
from skimage.transform import resize

# Preprocesamiento y Métricas
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn.pipeline import Pipeline

# --- OPTIMIZACIÓN BAYESIANA ---
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer

# --- CLASIFICADORES ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

In [ ]:
# Función para cargar imágenes
def load_image_dataset(root_dir, size=(64,64), max_per_class=None):
    X, y = [], []
    classes = sorted(os.listdir(root_dir))
    for cls in classes:
        cls_path = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_path): continue
        files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png','.jpg'))]
        if max_per_class: files = files[:max_per_class]
        for f in files:
            img = imread(os.path.join(cls_path, f))
            if img.ndim == 3: img = img[...,0]
            img_resized = resize(img, size, anti_aliasing=True)
            X.append(img_resized)
            y.append(cls)
    return np.array(X), np.array(y), classes

# Carga
X, y_str, classes = load_image_dataset("CMS_data", size=(64,64))
print("Datos cargados.")

# --- PREPROCESAMIENTO CRÍTICO ---
# 1. Normalizar X
X = X / X.max()
X_flat = X.reshape(len(X), -1)

# 2. Codificar etiquetas (Strings -> Enteros) para que la ANN no falle
le = LabelEncoder()
y_int = le.fit_transform(y_str)
print(f"Clases: {classes}")
print(f"Etiquetas convertidas: {np.unique(y_int)}")

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y_int, test_size=0.2, stratify=y_int, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Binarizar las etiquetas (necesario para ROC multiclase)
# Usamos un rango de enteros porque y_test ya son enteros
y_test_bin = label_binarize(y_test, classes=np.arange(len(classes)))
n_classes = y_test_bin.shape[1]

In [ ]:
# --- Función para graficar ROC ---
def plot_multiclass_roc(y_test_bin, y_score, classes, title):
    plt.figure(figsize=(8,6))
    for i in range(len(classes)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{classes[i]} (AUC = {roc_auc:.2f})")
    plt.plot([0,1], [0,1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

# --- Función para graficar Curvas de Aprendizaje ---
def plot_learning_curve(estimator, title, X, y, ylim=None, cv=3, n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)):
    plt.figure(figsize=(10, 6))
    plt.title(title)
    if ylim is not None: plt.ylim(*ylim)
    plt.xlabel("Ejemplos de entrenamiento")
    plt.ylabel("Score (Accuracy)")
    
    train_sizes_abs, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes, scoring="accuracy"
    )
    
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)
    
    plt.grid(True)
    plt.fill_between(train_sizes_abs, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes_abs, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes_abs, train_scores_mean, 'o-', color="r", label="Train Score")
    plt.plot(train_sizes_abs, test_scores_mean, 'o-', color="g", label="CV Score")
    plt.legend(loc="best")
    plt.show()

In [ ]:
%%time
print("Iniciando Optimización Bayesiana para ANN...")

# Pipeline para escalado + modelo
pipeline_ann = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=500, early_stopping=True, random_state=42))
])

search_space_ann = {
    'mlp__hidden_layer_sizes': Categorical([(50,), (100,), (50, 50), (100, 50)]),
    'mlp__alpha': Real(1e-5, 1e-1, prior='log-uniform'), # Regularización
    'mlp__learning_rate_init': Real(1e-4, 1e-1, prior='log-uniform')
}

bayes_ann = BayesSearchCV(
    pipeline_ann,
    search_space_ann,
    n_iter=10,
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

bayes_ann.fit(X_train, y_train)
best_ann = bayes_ann.best_estimator_
print(f"Mejores parámetros ANN: {bayes_ann.best_params_}")

In [ ]:
y_pred_ann = best_ann.predict(X_test)
y_score_ann = best_ann.predict_proba(X_test)

print("--- Reporte ANN ---")
print(classification_report(y_test, y_pred_ann, target_names=classes))

In [ ]:
# Matriz
cm = confusion_matrix(y_test, y_pred_ann)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap="Reds", xticks_rotation=45)
plt.title("Matriz de Confusión - ANN (Bayes)")
plt.show()

In [ ]:
# Precisión
report_dict = classification_report(y_test, y_pred_ann, output_dict=True, target_names=classes)
pd.DataFrame(report_dict).transpose().loc[classes, "precision"].plot(kind="bar", color="red")
plt.title("Precisión por clase - ANN")
plt.show()

In [ ]:
# ROC
plot_multiclass_roc(y_test_bin, y_score_ann, classes, "ROC - ANN (Bayes)")

In [ ]:
# Learning Curve
plot_learning_curve(best_ann, "Curva de Aprendizaje - ANN", X_train, y_train, cv=3, ylim=(0.4, 1.0))